# AI Evaluation

- AI Evaluation is the most critical part of getting an agent pilot into production. It gives a quantitative view into agent performance and enables a clear understanding of model performance that builds the trust with stakeholders and end users necessary to drive towards production.

- We will use built-in judges, custom guideline based judges, and a fully custom prompt based judge to evaluate performance

- This gives a full view into how the agent is performing, and the trace-level observational detail allows us to drill into specific examples to determine quality

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph
dbutils.library.restartPython()

In [ ]:
import json
from pathlib import Path
import mlflow

# Load configuration from setup notebook
CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
# Extract configuration variables
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
LABEL_SCHEMA_NAME = CONFIG["evaluation"]["label_schema_name"]
LABELING_SESSION_NAME = CONFIG["evaluation"]["labeling_session_name"]
ASSIGNED_USERS = CONFIG["evaluation"]["assigned_users"]
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

## Load Existing Evaluation Dataset or Create via FMAPI

First check whether the MLflow GenAI dataset (`DATASET_NAME`) exists and has records. If so, load those records directly for evaluation.

If no records are found, fall back to generating a new balanced question set via FMAPI.

In [ ]:
from mlflow.genai.datasets import get_dataset


def _extract_question_from_record(record: dict) -> str | None:
    """Best-effort extraction of the user question from a dataset record."""
    if not isinstance(record, dict):
        return None

    inputs = record.get("inputs")
    if isinstance(inputs, dict):
        input_messages = inputs.get("input")
        if isinstance(input_messages, list) and input_messages:
            first_msg = input_messages[0]
            if isinstance(first_msg, dict):
                content = first_msg.get("content")
                return content if isinstance(content, str) else None

    # Fallback shape seen in some trace-derived records
    request = record.get("request")
    if isinstance(request, dict):
        input_messages = request.get("input")
        if isinstance(input_messages, list) and input_messages:
            first_msg = input_messages[0]
            if isinstance(first_msg, dict):
                content = first_msg.get("content")
                return content if isinstance(content, str) else None

    return None


def _try_load_existing_eval_records(dataset_name: str) -> list[dict]:
    """Load existing dataset records directly if dataset exists."""
    try:
        eval_ds = get_dataset(name=dataset_name)
        eval_df = eval_ds.to_df()
    except Exception:
        return []

    eval_data = []
    for _, row in eval_df.iterrows():
        inputs = row.get("inputs")
        if inputs is None:
            continue

        if isinstance(inputs, str):
            inputs = json.loads(inputs)

        if isinstance(inputs, dict) and "request" in inputs and isinstance(inputs["request"], dict):
            # Trace-derived Responses schema: {"request": {"input": [...]}}
            request_obj = inputs["request"]
            msg_input = request_obj.get("input", [])
            if not isinstance(msg_input, list):
                msg_input = [msg_input]
            entry = {"inputs": {"input": msg_input}}
        elif isinstance(inputs, dict) and "input" in inputs:
            # Already in evaluate()-friendly shape
            msg_input = inputs["input"]
            if not isinstance(msg_input, list):
                msg_input = [msg_input]
            entry = {"inputs": {"input": msg_input}}
        else:
            msg_input = inputs if isinstance(inputs, list) else [inputs]
            entry = {"inputs": {"input": msg_input}}

        expectations = row.get("expectations")
        if expectations is not None:
            if isinstance(expectations, str):
                expectations = json.loads(expectations)
            if expectations:
                entry["expectations"] = expectations

        eval_data.append(entry)

    return eval_data


# 1) Reuse existing dataset if available
eval_dataset_records = _try_load_existing_eval_records(DATASET_NAME)

if eval_dataset_records:
    print(f"Loaded {len(eval_dataset_records)} evaluation records from dataset '{DATASET_NAME}'.")
else:
    print(f"No records found in dataset '{DATASET_NAME}'. Generating via FMAPI...")

    from openai import OpenAI
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    fmapi_client: OpenAI = w.serving_endpoints.get_open_ai_client()

    GENERATION_PROMPT = """You are an expert at generating evaluation examples for a baseball hitting analysis AI assistant.

The assistant helps batters prepare for matchups against specific pitchers. It has two kinds of tools:

## UC Functions (deterministic tools)
These answer specific, structured questions:
- lookup_player_by_name: Resolve player name to ID
- get_batter_pitcher_matchup: Historical pitches between specific batter-pitcher pairs
- get_pitcher_tendency_by_count: Pitch type/location by count (balls, strikes) and batter hand
- get_pitcher_tendency_with_runners: Same as above but filtered by base runner situation
- pitcher_embedding_lookup / pitcher_embedding_query: Find similar pitchers by pitch type
- batter_embedding_lookup / batter_embedding_query: Find similar batters
- pitcher_arsenal_lookup: Get all pitch types a pitcher throws
- recommend_batter_matchups_by_team: Best lineup matchups vs a pitcher
- get_team_batters: Full batter roster for a team
- batter_embeddings_for_ids: Get embedding vectors for multiple batters

## Genie Space (SQL-based analytics)
The assistant also has a Genie Space that can query underlying tables directly for questions the UC functions cannot answer:
- statcast_pitches: All pitch-level data
- dim_pitchers, dim_batters, dim_players: Player dimension tables
- pitcher_vectors_mean, batter_vectors_mean: Aggregated embedding vectors
- dim_pitcher_arsenal: Pitcher arsenal summary
- dim_batter_team_year, dim_pitcher_team_year: Team-season rosters

Generate exactly 30 diverse evaluation examples as a JSON array. Each example should be a single string (the user question).

IMPORTANT: Include a balanced mix of BOTH types:

### ~15 UC Function questions (answerable by the tools above):
- Specific batter vs pitcher matchup analysis
- Pitcher tendencies by count and batter hand (with specific count numbers like 0-2, 1-1, 3-2)
- Pitcher tendencies with runners on specific bases
- Finding similar pitchers/batters via embeddings
- Pitcher arsenal lookups
- Lineup construction against a specific pitcher
- Team roster queries

### ~15 Genie-only questions (require SQL analytics, NOT answerable by UC functions):
- Pitch distribution (% of each pitch type) for a specific team in a season
- Average velocity or spin rate across a team's pitching staff
- Which pitchers throw the hardest fastball in the league?
- What is the average launch angle for batters on a specific team?
- Compare pitch usage between two teams
- What percentage of pitches are breaking balls for a specific pitcher?
- Team-level batting statistics (avg exit velocity, barrel rate)
- Historical trends: how has a pitcher's velocity changed across seasons?
- Distribution of pitch locations (zone analysis) for a pitcher
- League-wide statistics and comparisons

Use real MLB player names and teams. Use the 3-letter team abbreviations: TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

Return ONLY a valid JSON array of strings, no other text.

Only ask for data from 2024 & 2025 seasons."""

    response = fmapi_client.chat.completions.create(
        model=CONFIG["llm"]["endpoint_name"],
        messages=[{"role": "user", "content": GENERATION_PROMPT}],
        temperature=0.8,
        max_tokens=4000,
    )

    raw_output = response.choices[0].message.content
    print("Raw FMAPI output (first 200 chars):")
    print(raw_output[:200])

In [ ]:
import re

# If we loaded existing records, just preview them.
# Otherwise parse FMAPI output and create records.
if eval_dataset_records:
    print(f"Reusing {len(eval_dataset_records)} evaluation records from existing dataset '{DATASET_NAME}'.\n")
    for i, rec in enumerate(eval_dataset_records):
        question = rec["inputs"]["input"][0]["content"]
        print(f"  {i+1:2d}. {question}")
else:
    json_match = re.search(r'\[.*\]', raw_output, re.DOTALL)
    if json_match:
        example_questions = json.loads(json_match.group())
    else:
        example_questions = json.loads(raw_output)

    print(f"Generated {len(example_questions)} evaluation questions\n")

    for i, q in enumerate(example_questions):
        print(f"  {i+1:2d}. {q}")

    eval_dataset_records = [
        {
            "inputs": {
                "input": [
                    {"role": "user", "content": question}
                ]
            }
        }
        for question in example_questions
    ]

print(f"\nPrepared {len(eval_dataset_records)} evaluation records")

## Define Judges / Scorers

We use three types of judges:
1. **RelevanceToQuery** - Built-in judge for response relevance
2. **Guidelines** - Custom guideline judge for baseball-specific language
3. **make_judge** - Fully custom judge for baseball analysis quality (1-5 Likert scale)

In [ ]:
import logging
logging.getLogger("mlflow.genai.judges.instructions_judge").setLevel(logging.ERROR)

from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
)
from agent import AGENT

# Guideline judge for baseball-specific language
baseball_language = "The response must use language that is appropriate for professional baseball players and coaches. It should reference baseball-specific terminology accurately."
baseball_language_judge = Guidelines(name="baseball_language", guidelines=baseball_language)

# Custom judge for baseball analysis quality (1-5 scale)
baseball_analysis_judge = make_judge(
    name=ALIGNED_JUDGE_NAME,
    instructions=(
        "Evaluate if the response in {{ outputs }} appropriately analyzes the available data and provides an actionable recommendation "
        "to the question in {{ inputs }}. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. "
        "Your grading criteria should be: "
        " 1: Completely unacceptable. Incorrect data interpretation or no recommendations"
        " 2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage"
        " 3: Somewhat acceptable. Relevant feedback provided with some strategic advantage"
        " 4: Mostly acceptable. Relevant feedback provided with strong strategic advantage"
        " 5: Completely acceptable. Relevant feedback provided with excellent strategic advantage"
    ),
    feedback_value_type=float,
    model=JUDGE_MODEL,  # Model used to evaluate (from config)
)

scorers = [RelevanceToQuery(), baseball_analysis_judge, baseball_language_judge]

# Register judge to experiment (skip if already registered)
try:
    registered_base_judge = baseball_analysis_judge.register(experiment_id=EXPERIMENT_ID)
    print(f"Registered base judge: {registered_base_judge.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Base judge '{ALIGNED_JUDGE_NAME}' already registered (OK)")
    else:
        print(f"Warning registering judge: {e}")

## Run Evaluation

In [ ]:
from agent import AGENT
from mlflow.genai import evaluate


def _eval_predict_fn(input):
    """Normalize evaluate() inputs into ResponsesAgentRequest input messages."""
    input_payload = input
    if isinstance(input_payload, dict) and "request" in input_payload and isinstance(input_payload["request"], dict):
        req_input = input_payload["request"].get("input", [])
        messages = req_input if isinstance(req_input, list) else [req_input]
    elif isinstance(input_payload, dict) and "input" in input_payload:
        req_input = input_payload["input"]
        messages = req_input if isinstance(req_input, list) else [req_input]
    elif isinstance(input_payload, list):
        messages = input_payload
    else:
        messages = [{"role": "user", "content": str(input_payload)}]

    return AGENT.predict({"input": messages})


results = evaluate(
    data=eval_dataset_records,
    predict_fn=_eval_predict_fn,
    scorers=scorers
)

In [ ]:
import numpy as np


def _extract_all_assessment_scores(run_id):
    """Collect numeric scores by assessment name from trace assessments."""
    traces_df = mlflow.search_traces(run_id=run_id)
    by_name = {}

    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            name = a.get("assessment_name") or "(unknown_assessment)"
            val = a.get("feedback", {}).get("value")
            score = None
            if val == "yes":
                score = 1.0
            elif val == "no":
                score = 0.0
            elif isinstance(val, (int, float)):
                score = float(val)

            if score is not None:
                by_name.setdefault(name, []).append(score)

    return by_name


print("=" * 60)
print("EVALUATION RESULTS SUMMARY")
print("=" * 60)

print("\n1) Raw metrics object:")
print(results.metrics if getattr(results, "metrics", None) else "(empty)")

print("\n2) Score summary (metrics + trace assessments):")
if isinstance(getattr(results, "metrics", None), dict) and results.metrics:
    for k, v in results.metrics.items():
        if isinstance(v, (int, float)):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
else:
    print("(No numeric metrics in results.metrics)")

assessment_scores = _extract_all_assessment_scores(results.run_id)
if assessment_scores:
    print("\nTrace assessment aggregates:")
    for name, scores in sorted(assessment_scores.items()):
        print(f"- {name}: mean={np.mean(scores):.4f} +/- {np.std(scores):.4f} (n={len(scores)})")
else:
    print("\nNo numeric trace assessments found.")

# Explicitly call out custom judge presence
custom_judge_present = ALIGNED_JUDGE_NAME in assessment_scores
print(f"\nCustom judge '{ALIGNED_JUDGE_NAME}' present: {custom_judge_present}")
if custom_judge_present:
    cj_scores = assessment_scores[ALIGNED_JUDGE_NAME]
    print(f"Custom judge mean: {np.mean(cj_scores):.4f} +/- {np.std(cj_scores):.4f} (n={len(cj_scores)})")

print("\n3) Run and table info:")
print(f"Run ID: {results.run_id}")
if hasattr(results, "tables") and results.tables:
    for tname, tdf in results.tables.items():
        print(f"- Table '{tname}': {len(tdf)} rows")

## Tag Successful Traces

Tag all traces with OK status as `eval: complete` for downstream alignment and optimization.

In [ ]:
# Grab the trace_id for all observations that have state of OK and apply the tag of eval: complete
ok_trace_ids = results.result_df.loc[results.result_df["state"] == "OK", "trace_id"]
print(f'Number of traces with OK status: {len(ok_trace_ids)}')

for trace_id in ok_trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="complete")

## Create GenAI Dataset from Traces

Collect the evaluated traces into an MLflow GenAI dataset for use in labeling sessions and judge alignment.

In [ ]:
from mlflow.genai.datasets import create_dataset, get_dataset

# Step 1: Create an empty evaluation dataset
try:
    eval_dataset = get_dataset(name=DATASET_NAME)
except Exception:
    eval_dataset = create_dataset(
        name=DATASET_NAME,
    )

print(f"Configured evaluation dataset: {eval_dataset.name}")

# Step 2: Grab all traces with tag "eval: complete" and add them to the dataset
print("\nSearching for traces with tag 'eval: complete'...")
traces_with_tag = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.eval = 'complete'",
    return_type="pandas"
)

print(f"Found {len(traces_with_tag)} traces with tag 'eval: complete'")

# Convert dataset to align with inputs needed for merge_traces()
if 'inputs' not in traces_with_tag.columns and 'request' in traces_with_tag.columns:
    print("Renaming 'request' column to 'inputs'...")
    traces_with_tag = traces_with_tag.rename(columns={'request': 'inputs'})

if 'outputs' not in traces_with_tag.columns and 'response' in traces_with_tag.columns:
    print("Renaming 'response' column to 'outputs'...")
    traces_with_tag = traces_with_tag.rename(columns={'response': 'outputs'})

eval_dataset = eval_dataset.merge_records(traces_with_tag)

## Create Label Schema and Labeling Session (Review App)

Set up the Review App for domain experts to provide human feedback on agent responses.
This enables:
- SME labeling of response quality (1-5 scale)
- Collection of human feedback for judge alignment (Pillar 6)
- Continuous improvement through human-in-the-loop evaluation

In [ ]:
from mlflow.genai import create_labeling_session, get_review_app
from mlflow.genai import label_schemas

# Step 1: Create label schemas for collecting feedback
# Create a custom schema that matches the baseball_analysis_judge criteria (1-5 scale)
baseball_analysis_schema = label_schemas.create_label_schema(
    name=LABEL_SCHEMA_NAME,
    type="feedback",
    title=LABEL_SCHEMA_NAME,
    input=label_schemas.InputNumeric(
        min_value=1.0,
        max_value=5.0,
    ),
    instruction=(
        "Evaluate if the response appropriately analyzes the available data and provides an actionable recommendation "
        "for the question. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. "
        "\n\n Your grading criteria should be: "
        "\n 1: Completely unacceptable. Incorrect data interpretation or no recommendations"
        "\n 2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage"
        "\n 3: Somewhat acceptable. Relevant feedback provided with some strategic advantage"
        "\n 4: Mostly acceptable. Relevant feedback provided with strong strategic advantage"
        "\n 5: Completely acceptable. Relevant feedback provided with excellent strategic advantage"
    ),
    enable_comment=True,  # Allow additional comments/feedback
    overwrite=True,
)

# Step 2: Set up the Review App with the deployed agent
AGENT_NAME = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
MODEL_NAME_SHORT = UC_MODEL_NAME.split('.')[-1]
review_app = get_review_app(experiment_id=EXPERIMENT_ID)

# Add the agent to the review app
review_app = review_app.add_agent(
    agent_name=MODEL_NAME_SHORT,
    model_serving_endpoint=AGENT_NAME,
    overwrite=True,
)

In [ ]:
# Step 3: Create the labeling session & add the traces for evaluation

labeling_session = create_labeling_session(
    name=f'{LABELING_SESSION_NAME}_sme',
    assigned_users=ASSIGNED_USERS,
    label_schemas=[LABEL_SCHEMA_NAME],  # Required: define what feedback to collect
)
# Add the dataset to the labeling session
labeling_session = labeling_session.add_dataset(
    dataset_name=DATASET_NAME
)
print(f"Created labeling session: {labeling_session.name}")
print(f"Labeling session ID: {labeling_session.labeling_session_id}")
print(f"Assigned users: {labeling_session.assigned_users}")
print(f"Labeling session URL: {labeling_session.url}")

## Next Steps

Once the labeling sessions are completed by SMEs, proceed to:
- **05-JudgeAlignment.ipynb** - Align the judge with SME feedback using MemAlign
- **06-PromptOptimization.ipynb** - Optimize the agent prompt using the aligned judge
- **07-AgentSkillsGeneration.ipynb** - Generate agent skills using the optimized prompt
- **08_create_agent_with_skills.ipynb** - Build the skills-enhanced agent
- **09-Evaluation.ipynb** - Compare both agents on a held-out dataset